# Notebook 4: MLP Deep Learning Model
This notebook implements the core deep learning model — a **Multi-Layer Perceptron (MLP)** — to predict road collision injury severity in Greater London. The model uses a triple-layered imbalance handling strategy:

1. **WeightedRandomSampler** — ensures Fatal crashes appear in every training batch
2. **Class Weights** — penalises Fatal misclassifications 160x harder than Slight
3. **Focal Loss** — dynamically concentrates learning effort on hard-to-classify crashes

The model must beat the Random Forest baseline: **Macro F1 > 0.31, Fatal Recall > 0%, MCC > 0.028**.

## Step 1: Load Data & Build PyTorch DataLoaders
We load the preprocessed artefacts from Notebook 2 and wrap them in PyTorch's `TensorDataset` (faster than a custom class for in-memory tabular data). The training DataLoader uses `WeightedRandomSampler` to oversample Fatal crashes — ensuring every batch of 64 contains minority class representation.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import joblib
import matplotlib.pyplot as plt
import warnings
import os
import copy

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device selection
device = torch.device('mps' if torch.backends.mps.is_available() else 
                       'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load preprocessed artefacts
data = joblib.load('../outputs/models/preprocessed_data.joblib')
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
class_weights = data['class_weights']
feature_names = data['feature_names']

print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Val:   {X_val.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Class weights: Fatal={class_weights[0]:.2f}, Serious={class_weights[1]:.2f}, Slight={class_weights[2]:.2f}")

# --- Convert to PyTorch Tensors ---
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# --- TensorDatasets ---
train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)
test_ds = TensorDataset(X_test_t, y_test_t)

# --- WeightedRandomSampler for training ---
# Assign each sample a weight based on its class (inverse frequency)
sample_weights = torch.tensor([class_weights[label] for label in y_train], dtype=torch.float32)
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # sample same number per epoch
    replacement=True  # required for oversampling minority classes
)

# --- DataLoaders ---
BATCH_SIZE = 64

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created (batch_size={BATCH_SIZE}):")
print(f"  Train: {len(train_loader)} batches with WeightedRandomSampler")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

# --- Verify sampler is working ---
# Check class distribution in a few training batches
class_counts = torch.zeros(3)
for batch_X, batch_y in train_loader:
    for c in range(3):
        class_counts[c] += (batch_y == c).sum().item()
    
class_pcts = class_counts / class_counts.sum() * 100
labels = ['Fatal', 'Serious', 'Slight']
print(f"\nSampler verification (effective class distribution across 1 epoch):")
for i, (name, pct) in enumerate(zip(labels, class_pcts)):
    print(f"  {name:>7s}: {pct:.1f}%  (original: {(y_train == i).sum() / len(y_train) * 100:.1f}%)")

Using device: mps


Train: 14,676 samples, 21 features
Val:   3,145 | Test: 3,146
Class weights: Fatal=64.37, Serious=2.00, Slight=0.40

DataLoaders created (batch_size=64):
  Train: 230 batches with WeightedRandomSampler
  Val:   50 batches
  Test:  50 batches

Sampler verification (effective class distribution across 1 epoch):
    Fatal: 33.2%  (original: 0.5%)
  Serious: 33.6%  (original: 16.7%)
   Slight: 33.1%  (original: 82.8%)


## Step 2: MLP Architecture
A 3-layer "funnel" MLP: 21 input features → 128 → 64 → 32 → 3 output classes. Each hidden layer follows the research-backed block pattern: `Linear → BatchNorm → ReLU → Dropout(0.3)`.

- **BatchNorm:** Stabilises the data flowing between layers so each layer receives numbers in a consistent range, even as the model's weights update during training.
- **ReLU:** Introduces non-linearity — without it, stacking layers is mathematically identical to a single layer.  
- **Dropout(0.3):** Randomly disables 30% of neurons each training step, forcing the model to build redundant knowledge and preventing overfitting on 14,676 samples.

In [2]:
class CollisionMLP(nn.Module):
    """
    Multi-Layer Perceptron for collision severity prediction.
    Architecture: 21 → 128 → 64 → 32 → 3 (funnel design)
    Each hidden block: Linear → BatchNorm → ReLU → Dropout
    """
    def __init__(self, input_dim=21, num_classes=3, dropout=0.3):
        super().__init__()
        
        self.network = nn.Sequential(
            # Layer 1: 21 → 128 (discover patterns from raw features)
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Layer 2: 128 → 64 (refine and compress patterns)
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Layer 3: 64 → 32 (distill into severity signals)
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Output: 32 → 3 (Fatal, Serious, Slight probabilities)
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialise the model
model = CollisionMLP(input_dim=X_train.shape[1], num_classes=3, dropout=0.3)
model = model.to(device)

# Display architecture summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("CollisionMLP Architecture:")
print(model)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Quick forward pass test
with torch.no_grad():
    sample_batch = X_train_t[:5].to(device)
    sample_output = model(sample_batch)
    probs = torch.softmax(sample_output, dim=1)
    print(f"\nForward pass test (5 samples):")
    print(f"  Raw logits shape: {sample_output.shape}")
    print(f"  Softmax probabilities (first sample):")
    print(f"    Fatal:   {probs[0, 0]:.4f}")
    print(f"    Serious: {probs[0, 1]:.4f}")
    print(f"    Slight:  {probs[0, 2]:.4f}")
    print(f"    Sum:     {probs[0].sum():.4f} (should be 1.0)")

CollisionMLP Architecture:
CollisionMLP(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=3, bias=True)
  )
)

Total parameters:     13,699
Trainable parameters: 13,699



Forward pass test (5 samples):
  Raw logits shape: torch.Size([5, 3])
  Softmax probabilities (first sample):
    Fatal:   0.4447
    Serious: 0.2906
    Slight:  0.2647
    Sum:     1.0000 (should be 1.0)
